### 1. Imports

In [ ]:
import pandas as pd
import glob
import os
from functools import reduce
import numpy as np

## 2. ETL

### 2.1 Estatística Educação Básica

In [ ]:
file_path = '/home/raimundoivy/Documents/pad_avaliação_02/dados/Sinopse_Estatistica_da_Educação_Basica_2024.xlsx'


In [ ]:
def etl_educacao_sheet_1_2(caminho_arquivo):
    df_1_2 = pd.read_excel(caminho_arquivo, sheet_name='1.2', skiprows=10, header=None)
    df_1_2.columns = [
        'Região Geográfica', 
        'Unidade da Federação', 
        'Município', 
        'Código do Município', 
        'Total Geral',
        'Urbana - Total',
        'Urbana - Federal',
        'Urbana - Estadual',
        'Urbana - Municipal',
        'Urbana - Privada',
        'Rural - Total',
        'Rural - Federal',
        'Rural - Estadual',
        'Rural - Municipal',
        'Rural - Privada'
    ]
    df_cleaned = df_1_2.copy()
    
    # Filtrar por código numérico do IBGE para excluir resumos por país/região e notas de rodapé incorporadas no Excel.
    df_cleaned = df_cleaned[pd.to_numeric(df_cleaned['Código do Município'], errors='coerce').notnull()]
    df_cleaned['Código do Município'] = df_cleaned['Código do Município'].astype(int)

    # Identificadores nos quais queremos ancorar o conjunto de dados
    id_vars = ['Região Geográfica', 'Unidade da Federação', 'Município', 'Código do Município']
    value_vars = [col for col in df_cleaned.columns if col not in id_vars]
    df_melted = df_cleaned.melt(id_vars=id_vars, value_vars=value_vars, var_name='Localizacao_Dependencia', value_name='Numero_Escolas')

    # Exemplo de manipulação de string: 'Urbana - Municipal' -> Localização: 'Urbana', Dependência: 'Municipal'
    # Tratamento de exceção: 'Total Geral' -> Localização: 'Total', Dependência: 'Total Geral'
    df_melted['Localizacao'] = df_melted['Localizacao_Dependencia'].apply(lambda x: x.split(' - ')[0] if ' - ' in x else 'Total')
    df_melted['Dependencia_Administrativa'] = df_melted['Localizacao_Dependencia'].apply(lambda x: x.split(' - ')[1] if ' - ' in x else 'Total Geral')
    
    df_melted = df_melted.drop(columns=['Localizacao_Dependencia'])

    # Limpar o formato numérico (hífens e pontos por NaN) + filtro medições vazias
    df_melted['Numero_Escolas'] = pd.to_numeric(df_melted['Numero_Escolas'], errors='coerce')
    df_melted = df_melted.dropna(subset=['Numero_Escolas'])
    df_melted['Numero_Escolas'] = df_melted['Numero_Escolas'].astype(int)
    
    # Reordenando e Renomeando
    df_final = df_melted[['Código do Município', 'Região Geográfica', 'Unidade da Federação', 'Município', 'Localizacao', 'Dependencia_Administrativa', 'Numero_Escolas']]
    df_final.columns = ['cod_ibge', 'regiao', 'uf', 'municipio', 'localizacao', 'dependencia', 'num_escolas']
    return df_final

In [ ]:
df_educacao_final = etl_educacao_sheet_1_2(file_path)
display(df_educacao_final.head(10))

In [ ]:
output_csv = '/home/raimundoivy/Documents/pad_avaliação_02/dados/educacao_basica_2024_1_2_cleaned.csv'
print(f"\nSaving to {output_csv}...")
df_educacao_final.to_csv(output_csv, index=False, encoding='utf-8')
print("Done!")

### 2.2 tabela6873 -- Tratores

In [ ]:
file_path = '/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6873.csv'

In [ ]:
def etl_tabela6873(caminho_arquivo):
    print("Loading raw CSV...")
    df_raw = pd.read_csv(
        caminho_arquivo, 
        sep=';', 
        skiprows=4, 
        engine='python',
        na_values=['-', 'X', '..', '...', ' '],
        on_bad_lines='skip',
        decimal=',' 
    )
    df_raw.columns = ['cod_ibge', 'localidade', 'condicao_produtor', 'cod_filtro', 'grupos_atividade', 'valor']
    df_raw = df_raw.dropna(subset=['cod_ibge'])
    
    # Filtrar apenas linhas que possuem Códigos IBGE numéricos
    df_raw['is_numeric_ibge'] = df_raw['cod_ibge'].astype(str).str.isnumeric()
    valid_rows = df_raw[df_raw['is_numeric_ibge']].copy()
    
    # Dividir o conjunto de dados ao meio (primeira metade: estabelecimentos, segunda metade: tratores)
    n_registros = len(valid_rows) // 2
    
    df_estab = valid_rows.iloc[:n_registros].copy()
    df_maq = valid_rows.iloc[n_registros:].copy()
    
    print("Processing numeric values...")
    df_estab['num_estabelecimentos'] = pd.to_numeric(df_estab['valor'], errors='coerce')
    df_maq['num_tratores'] = pd.to_numeric(df_maq['valor'], errors='coerce')
    
    # Nós devemos fazer o merge usando todas as chaves compostas identificadoras para garantir uma correspondência 1:1
    merge_keys = ['cod_ibge', 'localidade', 'condicao_produtor', 'grupos_atividade', 'cod_filtro']
    df_maq_slim = df_maq[merge_keys + ['num_tratores']]
    
    print("Merging variable blocks...")
    df_final = pd.merge(
        df_estab,
        df_maq_slim,
        on=merge_keys,
        how='left'
    )
    
    # Extrair UF e limpar Município do nome da localidade
    df_final['uf'] = df_final['localidade'].str.extract(r'\((.*?)\)$')[0]
    df_final['municipio'] = df_final['localidade'].str.replace(r'\s*\(.*?\)$', '', regex=True)
    
    # Calcular Nível Geo (ex: 1=Brasil, 2=Estado, 7=Cidade)
    df_final['geo_level'] = df_final['cod_ibge'].astype(str).str.len()
    
    # Remover colunas identificadoras desnecessárias resultantes do merge, E filtrar suas exclusões solicitadas
    cols_to_drop = [
        'condicao_produtor', 'grupos_atividade', 'geo_level', 
        'cod_filtro', 'valor', 'is_numeric_ibge', 'localidade'
    ]
    df_final = df_final.drop(columns=[c for c in cols_to_drop if c in df_final.columns], errors='ignore')
    
    # Reordenar as métricas analíticas restantes de forma organizada
    expected_cols = ['cod_ibge', 'uf', 'municipio', 'num_estabelecimentos', 'num_tratores']
    df_final = df_final[[c for c in expected_cols if c in df_final.columns]]
    
    # Finalizar a aplicação do tipo inteiro correto para cod_ibge
    df_final['cod_ibge'] = df_final['cod_ibge'].astype(int)
    
    return df_final

In [ ]:
df_6873_final = etl_tabela6873(file_path)
print(f"\nETL completed. Final shape: {df_6873_final.shape}")
display(df_6873_final.head())

In [ ]:
# Salvar na pasta de saída
output_csv = '/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6873_cleaned.csv'
print(f"\nSaving to {output_csv}...")
df_6873_final.to_csv(output_csv, index=False, encoding='utf-8')
print("Done!")

### 2.3 tabela6873 -- Área em Hectares

In [ ]:
file_path = '/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6773.csv'

In [ ]:
def etl_tabela6773(caminho_arquivo):
    print("Loading raw CSV...")
    # Carregar o arquivo bruto, ignorando as 4 primeiras linhas de metadados
    df_raw = pd.read_csv(
        caminho_arquivo, 
        sep=';', 
        skiprows=4, 
        engine='python',
        na_values=['-', 'X', '..', '...', ' '],
        on_bad_lines='skip',
        decimal=',' 
    )
    # Renomear colunas para nomes internos padronizados
    df_raw.columns = ['cod_ibge', 'localidade', 'finalidade', 'renda', 'cod_filtro', 'associacao', 'valor']
    # O conjunto de dados possui variáveis empilhadas: a primeira metade é 'Número de Estabelecimentos', a segunda metade é 'Área em Hectares'
    # Limpar as notas de rodapé vazias
    df_raw = df_raw.dropna(subset=['cod_ibge'])
    
    # Filtrar apenas linhas representando Códigos IBGE válidos
    df_raw['is_numeric_ibge'] = df_raw['cod_ibge'].astype(str).str.isnumeric()
    valid_rows = df_raw[df_raw['is_numeric_ibge']].copy()
    
    # Nós dividimos as linhas válidas pela metade: primeira metade = estabelecimentos, segunda metade = área
    n_registros = len(valid_rows) // 2
    
    df_estab = valid_rows.iloc[:n_registros].copy()
    df_area = valid_rows.iloc[n_registros:].copy()
    
    print("Processing columns...")
    df_estab['num_estabelecimentos'] = pd.to_numeric(df_estab['valor'], errors='coerce')
    df_area['area_hectares'] = pd.to_numeric(df_area['valor'], errors='coerce')
    
    # Extrair UF e limpar Município de 'localidade'
    df_estab['uf'] = df_estab['localidade'].str.extract(r'\((.*?)\)$')[0]
    df_estab['municipio'] = df_estab['localidade'].str.replace(r'\s*\(.*?\)$', '', regex=True)
    
    # Calcular Nível Geo para possível filtragem hierárquica (1 = Brasil, 7 = Cidade)
    df_estab['geo_level'] = df_estab['cod_ibge'].astype(str).str.len()
    
    # Remover colunas intermediárias que agora são redundantes
    df_estab = df_estab.drop(columns=['localidade', 'valor', 'is_numeric_ibge'])
    df_area = df_area[['cod_ibge', 'area_hectares']]
    
    print("Merging variable blocks...")
    # Remontar as duas variáveis como colunas lado a lado
    df_final = pd.merge(
        df_estab,
        df_area,
        on='cod_ibge',
        how='left'
    )
    
    # Reordenar o dataframe final de forma organizada
    df_final = df_final[['cod_ibge', 'geo_level', 'uf', 'municipio', 'finalidade', 'renda', 'associacao', 'num_estabelecimentos', 'area_hectares']]
    
    # Ajustar tipos
    df_final['cod_ibge'] = df_final['cod_ibge'].astype(int)
    
    return df_final

In [ ]:
df_6773_final = etl_tabela6773(file_path)
print(f"ETL completed. Final shape: {df_6773_final.shape}")
display(df_6773_final.head(30))

In [ ]:
# Formatos de saída
output_csv = '/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6773_cleaned.csv'
print(f"\nSaving to {output_csv}...")
df_6773_final.to_csv(output_csv, index=False, encoding='utf-8')
print("Done!")

### 2.4 tabela6873 -- Produtos / Área Colhida

In [ ]:
file_path = '/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela1612.csv'

In [ ]:
# Gerar os nomes corretos das colunas
years = range(1974, 2025)
products = [
    'Algodão herbáceo (em caroço)',
    'Cana-de-açúcar',
    'Milho (em grão)',
    'Soja (em grão)'
]
columns = ['cod_ibge', 'localidade']
for year in years:
    for prod in products:
        columns.append(f"{year}_{prod}")
print(f"Expected {len(columns)} columns.")

In [ ]:
# Tentar ler a primeira linha para ver quantas colunas realmente possui
with open(file_path, 'r', encoding='utf-8') as f:
    for _ in range(5):
        next(f)
    first_data_line = f.readline().strip()
num_cols_in_data = len(first_data_line.split(';'))
print(f"Data rows seem to have {num_cols_in_data} columns.")

In [ ]:
# Se houver um ponto e vírgula final extra, podemos ler uma coluna vazia extra
usecols = list(range(len(columns)))

In [ ]:
# Motor C rápido
df = pd.read_csv(
    file_path, 
    sep=';', 
    skiprows=5,
    names=columns,
    usecols=usecols,
    na_values=['-', 'X', '..', '...', ' '],
    decimal=',',
    engine='c', # Usar motor C para velocidade
    low_memory=False
)
print(f"Loaded CSV shape: {df.shape}")

In [ ]:
# Remover linhas que são apenas notas no final do CSV (cod_ibge deve ser numérico)
df = df[pd.to_numeric(df['cod_ibge'], errors='coerce').notnull()]
df['cod_ibge'] = df['cod_ibge'].astype(int)

In [ ]:
# Identificar níveis (Brasil=1, Região=1, Estado=2, Cidade=7 geralmente)
df['cod_str'] = df['cod_ibge'].astype(str)
df['geo_level'] = df['cod_str'].str.len()

In [ ]:
# Fazer o melt do dataframe para formato longo: cod_ibge, localidade, geo_level, ano, produto, area_colhida
id_vars = ['cod_ibge', 'localidade', 'geo_level']
value_vars = [col for col in columns if col not in ['cod_ibge', 'localidade']]
print("Melting dataframe...")
df_melted = df.melt(id_vars=id_vars, value_vars=value_vars, var_name='year_product', value_name='area_colhida_hectares')
print("Extracting year and product...")
df_melted[['ano', 'produto']] = df_melted['year_product'].str.extract(r'^(\d{4})_(.*)$')
df_melted.drop(columns=['year_product'], inplace=True)
df_melted['ano'] = df_melted['ano'].astype(int)
df_melted['area_colhida_hectares'] = pd.to_numeric(df_melted['area_colhida_hectares'], errors='coerce')
df_melted.dropna(subset=['area_colhida_hectares'], inplace=True)

In [ ]:
# Limpar nome da localidade
df_melted['uf'] = df_melted['localidade'].str.extract(r'\((.*?)\)$')[0]
df_melted['municipio'] = df_melted['localidade'].str.replace(r'\s*\(.*?\)$', '', regex=True)

In [ ]:
# Reordenar colunas
df_final = df_melted[['cod_ibge', 'geo_level', 'uf', 'municipio', 'ano', 'produto', 'area_colhida_hectares']]
print(f"ETL completed. Final shape: {df_final.shape}")
display(df_final.head(30))

In [ ]:
# Salvar em um novo CSV
output_csv = '/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela1612_cleaned.csv'
print(f"Saving to {output_csv}...")
df_final.to_csv(output_csv, index=False, encoding='utf-8')
print("Done!")

### 2.4 IBGE População

In [ ]:
file_path = '/home/raimundoivy/Documents/pad_avaliação_02/dados/br_ibge_populacao_municipio.csv'

In [ ]:
def etl_populacao(caminho):
    print("Loading population data...")
    df = pd.read_csv(caminho, sep=',')
    
    print(f"Original shape: {df.shape}")
    print("Years available:", sorted(df['ano'].unique()))
    
    # Limpando os tipos de dados para evitar problemas de conversão de object/float nos JOINS subsequentes
    df['id_municipio'] = pd.to_numeric(df['id_municipio'], errors='coerce')
    df = df.dropna(subset=['id_municipio'])
    df['id_municipio'] = df['id_municipio'].astype(int)
    
    # Renomear chaves para corresponder universalmente ao resto de nossos pipelines
    df = df.rename(columns={'id_municipio': 'cod_ibge', 'sigla_uf': 'uf'})
    
    return df

In [ ]:
df_pop_final = etl_populacao(file_path)
print(f"\nETL completed. Final shape: {df_pop_final.shape}")
display(df_pop_final.head(10))

In [ ]:
# Salvar saída
output_csv = '/home/raimundoivy/Documents/pad_avaliação_02/dados/br_ibge_populacao_municipio_cleaned.csv'
print(f"\nSaving to {output_csv}...")
df_pop_final.to_csv(output_csv, index=False, encoding='utf-8')
print("Done!")

## 3. Cruzando os Dados

In [ ]:
# Carregar todos os conjuntos de dados limpos que construímos!
print("Loading individual datasets...")
df_pop = pd.read_csv('/home/raimundoivy/Documents/pad_avaliação_02/dados/br_ibge_populacao_municipio_cleaned.csv')
df_edu = pd.read_csv('/home/raimundoivy/Documents/pad_avaliação_02/dados/educacao_basica_2024_1_2_cleaned.csv')
df_1612 = pd.read_csv('/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela1612_cleaned.csv')
df_6773 = pd.read_csv('/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6773_cleaned.csv')
df_6873 = pd.read_csv('/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6873_cleaned.csv')
print("Applying cross-reference logic via 'cod_ibge'...")

In [ ]:
# 1. Tabela Base: Usamos a métrica de População de 2024 como âncora já que valida todos os 5.570 códigos IBGE ativos reais
base = df_pop[df_pop['ano'] == 2024][['cod_ibge', 'uf', 'populacao']].copy()
base = base.rename(columns={'populacao': 'populacao_2024'})

In [ ]:
# 2. Adicionar Dados de Educação
# O conjunto de dados possui múltiplas linhas por localizacao/dependencia, então nós agrupamos seguramente apenas por código IBGE para obter o total de escolas dentro da fronteira municipal
edu_agg = df_edu.groupby('cod_ibge')['num_escolas'].sum().reset_index()
base = pd.merge(base, edu_agg, on='cod_ibge', how='left')

In [ ]:
# 3. Adicionar Área Agrícola (Tabela 1612 - Histórico)
# Vamos filtrar pelo ano estatístico mais recente registrado (2024 ou 2023 dependendo da cultura)
max_ano = df_1612['ano'].max()
df_1612_recent = df_1612[df_1612['ano'] == max_ano]

In [ ]:
# Pivotar os produtos agrícolas dinamicamente para termos uma linha por município, e produtos distintos agindo como colunas simples horizontais
agri_pivot = df_1612_recent.pivot_table(
    index='cod_ibge', 
    columns='produto', 
    values='area_colhida_hectares', 
    aggfunc='sum'
).reset_index()

In [ ]:
# Renomear strings do cabeçalho para snake_case evitando espaços
agri_pivot.columns = ['cod_ibge'] + [f"area_{c.split(' (')[0].replace('-', '_').replace(' ', '_').lower()}_{max_ano}" for c in agri_pivot.columns if c != 'cod_ibge']
base = pd.merge(base, agri_pivot, on='cod_ibge', how='left')

In [ ]:
# 4. Adicionar Área de Estabelecimentos Agropecuários (Tabela 6773)
# Para evitar a dupla correspondência dos cabeçalhos de Estado/Região, vamos filtrar rigorosamente o geo_level apenas para '7' (nível Cidade)
if 'geo_level' in df_6773.columns:
    df_6773_mun = df_6773[df_6773['geo_level'] == 7].drop_duplicates(subset=['cod_ibge'])
else:
    df_6773_mun = df_6773.drop_duplicates(subset=['cod_ibge'])
df_6773_mun = df_6773_mun[['cod_ibge', 'num_estabelecimentos', 'area_hectares']]
df_6773_mun.columns = ['cod_ibge', 'estabelecimentos_agro_total', 'area_agro_total_hectares']
base = pd.merge(base, df_6773_mun, on='cod_ibge', how='left')

In [ ]:
# 5. Adicionar Dados de Uso de Tratores (Tabela 6873)
df_6873_mun = df_6873.drop_duplicates(subset=['cod_ibge'])
df_6873_mun = df_6873_mun[['cod_ibge', 'num_tratores']]
base = pd.merge(base, df_6873_mun, on='cod_ibge', how='left')
print(f"\nFinal Unified Data shape (1 row per municipality): {base.shape}")
display(base.head(30))

In [ ]:
# Salvar conjunto de dados Master de Referência
output_master_csv = '/home/raimundoivy/Documents/pad_avaliação_02/dados/master_municipios_agrupado.csv'
print(f"\nSaving to {output_master_csv} ...")
base.to_csv(output_master_csv, index=False, encoding='utf-8')
print("Complete!")

In [ ]:
def processar_polars(file_path, lista_arquivos, arquivo_saida, cols):
    tempo_inicio = time.time()
    caminhos_completos = [os.path.join(file_path, f) for f in lista_arquivos]
    print(f'inicio do processo de consolidação de {len(lista_arquivos)}')
    lf = pl.scan_csv(caminhos_completos)
    lf = lf.select(cols)
    print('processando e salvando...')

    with tqdm (total=1, desc='progresso de consolidação', colour='blue') as pbar:
        lf.sink_csv(
            arquivo_saida,
            )
        pbar.update(1)

    tempo_total = time.time() - tempo_inicio
    linhas_finais = pl.read_csv(arquivo_saida).shape[0]
    print(f'\n processo finalizado {arquivo_saida}')
    print(f'arquivo gerado: {arquivo_saida}')